# Petőfi-GPT – tanítás Google Colabon (GPU)

Ez a notebook a repó kódjával tanítja a modelleket GPU-n. **Semmit nem kell átírni a kódban:** ugyanaz a `train.py` fut, mint a saját gépeden, csak a `--preset colab` kapcsolóval.

**Előtte:** *Runtime → Change runtime type → T4 GPU* (vagy bármilyen elérhető GPU), majd *Save*.

| Lépés | Mit csinál |
|---|---|
| 1 | ellenőrzi a GPU-t |
| 2 | letölti (klónozza) a repót a GitHubról |
| 3 | csatlakoztatja a Google Drive-ot, hogy a checkpointok megmaradjanak |
| 4 | előállítja az adatot (`prepare_data.py`), és ellenőrzi az ujjlenyomatát |
| 5 | gyors próbák: modell-önteszt + 50 lépéses GPU-tanítás |
| 6 | **a fő tanítás:** `python train.py --preset colab` |
| 7 | mintaszöveg az új modellből |
| 8 | további futások (ablationök, később a DIFF) |

## 1. GPU ellenőrzése

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA elérhető:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Nincs GPU: Runtime -> Change runtime type -> T4 GPU"

## 2. A repó letöltése (vagy frissítése, ha már letöltötted)

In [ ]:
import os
%cd /content
if not os.path.exists("petofi-gpt"):
    !git clone https://github.com/janosibenedek/petofi-gpt.git
%cd /content/petofi-gpt
!git pull
!git log --oneline -3

## 3. Google Drive: hogy a checkpointok megmaradjanak

A Colab gépe ideiglenes: ha a munkamenet megszakad, minden fájl elveszik róla. A checkpointokat ezért a Google Drive-odra mentjük (`MyDrive/petofi-gpt/checkpoints`). A `train.py` minden javuláskor ment, így egy megszakadt futásból is megmarad az addigi legjobb modell.

Ha nem akarod a Drive-ot használni, állítsd `USE_DRIVE = False`-ra. Ekkor a fájlok a Colab gépén maradnak, és a végén le kell töltened őket (8. lépés).

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/petofi-gpt/checkpoints"
else:
    OUT = "checkpoints"
os.makedirs(OUT, exist_ok=True)
print("checkpointok helye:", OUT)

## 4. Az adat előállítása

Ugyanaz a szkript, mint a gépeden. Az ujjlenyomat (SHA-256) igazolja, hogy pontosan ugyanabból a forrásfájlból dolgozik, így a train/val/test felosztás is ugyanaz.

Ha `False` jönne ki (a Gutenberg időközben módosította a fájlt), töltsd fel a saját gépeden lévő `data/raw/pg41504.txt`-t a Colab bal oldali *Files* paneljén, és futtasd így: `!python prepare_data.py --raw-file pg41504.txt`.

In [ ]:
!python prepare_data.py
import json
meta = json.load(open("data/meta.json", encoding="utf-8"))
EXPECTED = "aa7655a76f0321eaf42d2a56f97b9ddbc7af9274ece654f36d3d3a119c5bffdf"
print("ujjlenyomat egyezik a helyi futással:", meta["raw_sha256"] == EXPECTED)
print({k: v for k, v in meta["splits"].items()})

## 5. Gyors próbák

1. A modell öntesztje (ugyanaz, mint a gépeden).
2. **50 lépéses próbatanítás GPU-n**, a Colab-presettel. Ez ellenőrzi, hogy működik-e a GPU-s ág (16 bites *mixed precision*), és megmutatja, mennyi ideig tart egy lépés. Ebből becsülhető a teljes futás ideje.

In [ ]:
!python model.py
# the smoke-test checkpoint is not needed later: it stays on the temporary Colab disk, not on Drive
!python train.py --preset colab --max-iters 50 --eval-interval 25 --eval-iters 5 --name colab_smoke --out-dir checkpoints

## 6. A fő tanítás

A Colab-preset: 6 blokk, 384 dimenzió, 6 fej, 256 karakteres kontextus, dropout 0,2, kb. 10,8 millió paraméter. Tanítás: batch 64, lr 1e-3, legfeljebb 6000 lépés, early stopping a val loss alapján.

**Várható idő:** a 5. lépés kiírt idejéből becsülhető (kb. a 25 lépés ideje × 240). T4-en nagyjából fél–egy óra. A log 200 lépésenként ír, a `* best, saved` jelzi a mentést a Drive-ra.

In [ ]:
!python train.py --preset colab --out-dir "{OUT}"

## 7. Mintaszöveg az új modellből

In [ ]:
!python generate.py --checkpoint "{OUT}/colab.pt" --prompt "Szabadság, szerelem!\n" --temperature 0.5 0.8 1.0 --max-new-tokens 400

## 8. Letöltés a saját gépedre

Ha a Drive-ot használtad, a fájl ott van: *Google Drive → petofi-gpt → checkpoints → colab.pt*. Innen letöltheted, és a gépeden a `checkpoints/` mappába teheted.

Ha nem használtad a Drive-ot, ez a cella letölti a böngészővel:

In [ ]:
from google.colab import files
files.download(f"{OUT}/colab.pt")

## 9. További futások (később, egyenként)

Mindegyik ugyanazt a `train.py`-t futtatja, egyetlen különbséggel. A checkpoint neve automatikusan következik a kapcsolókból.

| Futás | Parancs | Checkpoint |
|---|---|---|
| ablation: ReLU | `--activation relu` | `colab_relu.pt` |
| ablation: residual nélkül | `--no-residual` | `colab_nores.pt` |
| (opcionális) hiperparaméter: dropout 0,1 | `--dropout 0.1 --name colab_drop01` | `colab_drop01.pt` |
| fő kísérlet: Differential Transformer | (a DIFF attention elkészülte után) | `colab_diff.pt` |

A cellák elején lévő `#` jelet töröld ki annál a sornál, amelyiket futtatni szeretnéd.

In [ ]:
# !python train.py --preset colab --activation relu --out-dir "{OUT}"
# !python train.py --preset colab --no-residual --out-dir "{OUT}"
# !python train.py --preset colab --dropout 0.1 --name colab_drop01 --out-dir "{OUT}"